In [54]:
# load the datapath that has been cleaned and preprocessed 

import pandas as pd

# Dataset paths
DATA_PATHS = {
    "train": "/home/gibannn/kuliah/sem3/paper/SMILES2VEC/data/AHR/train_data_class_weight_70.xlsx",
    "test":  "/home/gibannn/kuliah/sem3/paper/SMILES2VEC/data/AHR/test_data_class_weight_15.xlsx",
    "val":   "/home/gibannn/kuliah/sem3/paper/SMILES2VEC/data/AHR/val_data_class_weight_15.xlsx"
}

print("Datapaths loaded\n")


def load_dataset(path, name):
    """Load dataset and split into features (X) and labels (y)."""
    
    df = pd.read_excel(path)
    
    X = df.drop(columns="label")
    y = df["label"]

    print(f"{name.upper()} DATA")
    print(f"X_{name} shape: {X.shape} | y_{name} shape: {y.shape}")
    
    label_dist = y.value_counts().rename(index={0: "Label 0", 1: "Label 1"})
    print("Label distribution:\n", label_dist, "\n")

    return X, y


# Load datasets
X_train, y_train = load_dataset(DATA_PATHS["train"], "train")
X_test, y_test   = load_dataset(DATA_PATHS["test"], "test")
X_val, y_val     = load_dataset(DATA_PATHS["val"], "val")


Datapaths loaded

TRAIN DATA
X_train shape: (4701, 253) | y_train shape: (4701,)
Label distribution:
 label
Label 0    4168
Label 1     533
Name: count, dtype: int64 

TEST DATA
X_test shape: (1008, 253) | y_test shape: (1008,)
Label distribution:
 label
Label 0    894
Label 1    114
Name: count, dtype: int64 

VAL DATA
X_val shape: (1007, 253) | y_val shape: (1007,)
Label distribution:
 label
Label 0    893
Label 1    114
Name: count, dtype: int64 



In [57]:
vocab_size = int(
    max(
        X_train.to_numpy().max(),
        X_test.to_numpy().max(),
        X_val.to_numpy().max()
    )
)

# Maximum sequence length
MAX_LEN = X_train.shape[1]

print("Dataset Statistics")
print(f"Vocabulary size: {vocab_size}")
print(f"Max sequence length: {MAX_LEN}")

print("\nClass distribution in training set (class weights):")
print(
    y_train.value_counts()
    .rename(index={
        0: "Label 0 (Negative)",
        1: "Label 1 (Positive)"
    })
)

print("\nClass distribution in test set:")
print(
    y_test.value_counts()
    .rename(index={
        0: "Label 0 (Negative)",
        1: "Label 1 (Positive)"
    })
)

print("\nClass distribution in validation set:")
print(
    y_val.value_counts()
    .rename(index={
        0: "Label 0 (Negative)",
        1: "Label 1 (Positive)"
    })
)

Dataset Statistics
Vocabulary size: 111
Max sequence length: 253

Class distribution in training set (class weights):
label
Label 0 (Negative)    4168
Label 1 (Positive)     533
Name: count, dtype: int64

Class distribution in test set:
label
Label 0 (Negative)    894
Label 1 (Positive)    114
Name: count, dtype: int64

Class distribution in validation set:
label
Label 0 (Negative)    893
Label 1 (Positive)    114
Name: count, dtype: int64


In [58]:
# ================================
# Environment Configuration
# ================================
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"       # Suppress TensorFlow logs
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # Use GPU 0 and 1
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"      # Disable OneDNN optimizations


# ================================
# Imports
# ================================
import warnings
import random
import time

import numpy as np
import tensorflow as tf
import sklearn.metrics as metrics

warnings.filterwarnings("ignore", category=FutureWarning)


# ================================
# Reproducibility
# ================================
RANDOM_STATE = 116

def set_seeds(seed: int = 116):
    """Set seeds for reproducibility."""
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)


set_seeds(RANDOM_STATE)


# ================================
# GPU Configuration
# ================================
def setup_gpu():
    """Configure TensorFlow GPU memory growth."""
    
    gpus = tf.config.list_physical_devices("GPU")

    if not gpus:
        print("No GPU detected. Using CPU.")
        return

    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        logical_gpus = tf.config.list_logical_devices("GPU")

        print(f"GPUs available: {len(gpus)} physical | {len(logical_gpus)} logical")

    except RuntimeError as e:
        print("GPU setup failed:", e)


setup_gpu()


# ================================
# Training Configuration
# ================================
def set_training_config(batch_size=256, iterations=128, epochs=50):
    """Define training hyperparameters."""
    
    config = {
        "BATCH_SIZE": batch_size,
        "ITERATIONS": iterations,
        "EPOCHS": epochs
    }

    return config


TRAIN_CONFIG = set_training_config()

BATCH_SIZE = TRAIN_CONFIG["BATCH_SIZE"]
ITERATIONS = TRAIN_CONFIG["ITERATIONS"]
EPOCHS = TRAIN_CONFIG["EPOCHS"]

print(f"Training config -> Batch: {BATCH_SIZE} | Iterations: {ITERATIONS} | Epochs: {EPOCHS}")

No GPU detected. Using CPU.
Training config -> Batch: 256 | Iterations: 128 | Epochs: 50


In [64]:
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,GRU, Bidirectional,
    Dense, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import plot_model
from sklearn.utils.class_weight import compute_class_weight


weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(weights))
# Paths (simple and portable)

MODEL_DIR = "./model"
IMAGE_DIR = "./image"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)

MODEL_FILE = os.path.join(MODEL_DIR, "best_bigru_conv_model.h5")
MODEL_IMAGE = os.path.join(IMAGE_DIR, "bigru_conv_architecture.png")



# Model Architecture

def build_bigru_conv_model(sequence_length, vocabulary_size):

    set_seeds(RANDOM_STATE)

    model = Sequential([

        # Embedding
        Embedding(
            input_dim=vocabulary_size + 1,
            output_dim=128,
            mask_zero=False
        ),

        # Bidirectional GRU
        Bidirectional(
            GRU(
                32,
                return_sequences=False,
                dropout=0.2,
                recurrent_dropout=0.2
            )
        ),

        # Output layer
        Dense(1, activation="sigmoid")
    ])

    return model



# Build Model

model = build_bigru_conv_model(MAX_LEN, vocab_size)

loss = tf.keras.losses.BinaryFocalCrossentropy(gamma=2,alpha=0.25)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss=loss,
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")

    ]
)

model.build(input_shape=(None, MAX_LEN))

print("Model built successfully")
model.summary()



# Callbacks

early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=1,
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    filepath=MODEL_FILE,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)


# Save Model Architecture Diagram
plot_model(
    model,
    to_file=MODEL_IMAGE,
    show_shapes=True,
    show_layer_names=True
)

print(f"Model diagram saved to: {MODEL_IMAGE}")

Model built successfully


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 253, 128)       │        14,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_8 (Bidirectional) │ (None, 64)             │        31,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45,505 (177.75 KB)

 Trainable params: 45,505 (177.75 KB)

 Non-trainable params: 0 (0.00 B)

Model diagram saved to: ./image/bigru_conv_architecture.png


In [61]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

def compute_classification_metrics(y_true, y_pred):
    """
    Returns TP, TN, FP, FN, Precision, Recall, F1, Accuracy
    """
    cm = confusion_matrix(y_true, y_pred)
    
    tn, fp, fn, tp = cm.ravel()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    f1        = f1_score(y_true, y_pred, zero_division=0)
    accuracy  = accuracy_score(y_true, y_pred)

    return {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Accuracy": accuracy
    }

print("the function to compute classification metrics have been defined")

the function to compute classification metrics have been defined


In [68]:
import os
import time
import pandas as pd
import numpy as np
import tensorflow as tf
from datetime import timedelta
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


def to_numpy(x):
    """Convert pandas objects to numpy arrays."""
    return x.values if hasattr(x, "values") else x


def train_and_evaluate_model(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    epochs=100,
    batch_size=128,
    learning_rate=3e-4,
    model_name="BiGRU+CNN",
    model_save_path="./model/best_model.keras",
    patience=5,
    verbose=1,
    weights=None
):
    """
    Train and evaluate sequence classification model.
    """

    print(f"TRAINING {model_name}")
    print("=" * 50)

    start_time = time.time()

    # Convert datasets
    X_train, y_train = to_numpy(X_train), to_numpy(y_train)
    X_val, y_val = to_numpy(X_val), to_numpy(y_val)
    X_test, y_test = to_numpy(X_test), to_numpy(y_test)

    
    # Build model
    
    print("\nBuilding model...")
    model = build_bigru_conv_model(MAX_LEN, vocab_size)

    
    # Compile
    
    print("[2/4] Compiling model...")

    loss_fn = tf.keras.losses.BinaryFocalCrossentropy(gamma=2)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss=loss_fn,
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    model.build(input_shape=(None, MAX_LEN))

    if verbose:
        model.summary()

    
    # Callbacks
    
    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)

    callbacks = [

        EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=patience,
            restore_best_weights=True,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=model_save_path,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1
        )
    ]

    
    # Training
    
    print("\nTraining model...")
    print(f"Epochs: {epochs}")
    print(f"Batch size: {batch_size}")
    print(f"Training samples: {len(X_train)}")

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        class_weight=weights,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=verbose
    )

    training_time = time.time() - start_time

    
    # Evaluation
    
    print("\nEvaluating model...")

    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int)

    metrics_dict = compute_classification_metrics(y_test, y_pred)

    results_df = pd.DataFrame([{
        "Model": model_name,
        "Accuracy": metrics_dict["Accuracy"],
        "Precision": metrics_dict["Precision"],
        "Recall": metrics_dict["Recall"],
        "F1": metrics_dict["F1-Score"],
        "TP": metrics_dict["TP"],
        "TN": metrics_dict["TN"],
        "FP": metrics_dict["FP"],
        "FN": metrics_dict["FN"],
        "Training_Time": str(timedelta(seconds=int(training_time))),
        "Epochs": len(history.history["loss"])
    }])

    print("\nTRAINING COMPLETE")
    print("=" * 50)
    print(results_df.to_string(index=False))

    return model, history, results_df

In [69]:
model, history, results_df = train_and_evaluate_model(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    weights=class_weights,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=0.001,
    model_name="BiGRU",
    model_save_path=MODEL_FILE,
    patience=1,
    verbose=1
)

TRAINING BiGRU

Building model...
[2/4] Compiling model...


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 253, 128)       │        14,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_9 (Bidirectional) │ (None, 64)             │        31,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45,505 (177.75 KB)

 Trainable params: 45,505 (177.75 KB)

...

19/19 ━━━━━━━━━━━━━━━━━━━━ 16s 408ms/step - accuracy: 0.5441 - auc: 0.5843 - loss: 0.1706 - val_accuracy: 0.6365 - val_auc: 0.6807 - val_loss: 0.1662
Epoch 2/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 284ms/step - accuracy: 0.5509 - auc: 0.6769 - loss: 0.1675
Epoch 2: val_auc improved from 0.68067 to 0.69665, saving model to ./model/best_bigru_conv_model.h5


19/19 ━━━━━━━━━━━━━━━━━━━━ 6s 307ms/step - accuracy: 0.6050 - auc: 0.6827 - loss: 0.1618 - val_accuracy: 0.6892 - val_auc: 0.6966 - val_loss: 0.1495
Epoch 3/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step - accuracy: 0.6225 - auc: 0.7136 - loss: 0.1604
Epoch 3: val_auc improved from 0.69665 to 0.72809, saving model to ./model/best_bigru_conv_model.h5


19/19 ━━━━━━━━━━━━━━━━━━━━ 6s 296ms/step - accuracy: 0.6539 - auc: 0.7289 - loss: 0.1542 - val_accuracy: 0.6614 - val_auc: 0.7281 - val_loss: 0.1532
Epoch 4/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step - accuracy: 0.6409 - auc: 0.7551 - loss: 0.1529
Epoch 4: val_auc improved from 0.72809 to 0.75176, saving model to ./model/best_bigru_conv_model.h5


19/19 ━━━━━━━━━━━━━━━━━━━━ 5s 243ms/step - accuracy: 0.6433 - auc: 0.7658 - loss: 0.1469 - val_accuracy: 0.6336 - val_auc: 0.7518 - val_loss: 0.1562
Epoch 5/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 278ms/step - accuracy: 0.6665 - auc: 0.7749 - loss: 0.1465
Epoch 5: val_auc improved from 0.75176 to 0.75921, saving model to ./model/best_bigru_conv_model.h5


19/19 ━━━━━━━━━━━━━━━━━━━━ 6s 300ms/step - accuracy: 0.6730 - auc: 0.7848 - loss: 0.1407 - val_accuracy: 0.6733 - val_auc: 0.7592 - val_loss: 0.1422
Epoch 6/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 296ms/step - accuracy: 0.6631 - auc: 0.7755 - loss: 0.1477
Epoch 6: val_auc did not improve from 0.75921
19/19 ━━━━━━━━━━━━━━━━━━━━ 6s 321ms/step - accuracy: 0.6690 - auc: 0.7800 - loss: 0.1423 - val_accuracy: 0.5968 - val_auc: 0.7583 - val_loss: 0.1627
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 5.

Evaluating model...

TRAINING COMPLETE
Model  Accuracy  Precision   Recall       F1  TP  TN  FP  FN Training_Time  Epochs
BiGRU   0.65873     0.2125 0.745614 0.330739  85 579 315  29       0:00:44       6


In [70]:
y_pred_prob = model.predict(X_train, verbose=0)
y_pred = (y_pred_prob > 0.3).astype(int)
metrics_dict = compute_classification_metrics(y_train, y_pred)

In [71]:
results_df = pd.DataFrame([{
    "Accuracy": metrics_dict["Accuracy"],
    "Precision": metrics_dict["Precision"],
    "Recall": metrics_dict["Recall"],
    "F1": metrics_dict["F1-Score"],
    "TP": metrics_dict["TP"],
    "TN": metrics_dict["TN"],
    "FP": metrics_dict["FP"],
    "FN": metrics_dict["FN"],
}])

print("\nTRAINING DATA METRICS")
print("=" * 50)
print(results_df.to_string(index=False))
print("=" * 50)


TRAINING DATA METRICS
 Accuracy  Precision   Recall       F1  TP  TN   FP  FN
 0.267603   0.133317 0.992495 0.235059 529 729 3439   4


In [72]:
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = (y_pred_prob > 0.4).astype(int)

metrics_dict = compute_classification_metrics(y_test, y_pred)

In [73]:
results_df = pd.DataFrame([{
    "Accuracy": metrics_dict["Accuracy"],
    "Precision": metrics_dict["Precision"],
    "Recall": metrics_dict["Recall"],
    "F1": metrics_dict["F1-Score"],
    "TP": metrics_dict["TP"],
    "TN": metrics_dict["TN"],
    "FP": metrics_dict["FP"],
    "FN": metrics_dict["FN"],
}])

print("\nTRAINING DATA METRICS")
print("=" * 50)
print(results_df.to_string(index=False))
print("=" * 50)


TRAINING DATA METRICS
 Accuracy  Precision   Recall      F1  TP  TN  FP  FN
 0.433532   0.163476 0.973684 0.27995 111 326 568   3


In [74]:
model, history, results = train_and_evaluate_model(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=0.001,
    model_name="BiGRU",
    model_save_path="./model/best_bigru_model.keras",
    patience=5,
    verbose=1,
    weights=class_weights
)

TRAINING BiGRU

Building model...
[2/4] Compiling model...


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 253, 128)       │        14,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_10                │ (None, 64)             │        31,104 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45,505 (177.75 KB)

 Trainable params: 45,505 (177.75 KB)

 Non-trainable params: 0 (0.00 B)


Training model...
Epochs: 50
Batch size: 256
Training samples: 4701
Epoch 1/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - accuracy: 0.3959 - auc: 0.5465 - loss: 0.1759
Epoch 1: val_auc improved from None to 0.68067, saving model to ./model/best_bigru_model.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 15s 407ms/step - accuracy: 0.5441 - auc: 0.5843 - loss: 0.1706 - val_accuracy: 0.6365 - val_auc: 0.6807 - val_loss: 0.1662
Epoch 2/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 285ms/step - accuracy: 0.5509 - auc: 0.6769 - loss: 0.1675
Epoch 2: val_auc improved from 0.68067 to 0.69665, saving model to ./model/best_bigru_model.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 6s 307ms/step - accuracy: 0.6050 - auc: 0.6827 - loss: 0.1618 - val_accuracy: 0.6892 - val_auc: 0.6966 - val_loss: 0.1495
Epoch 3/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 282ms/step - accuracy: 0.6225 - auc: 0.7136 - loss: 0.1604
Epoch 3: val_auc improved from 0.69665 to 0.72809, saving model to ./model/best_bigru_model.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 6s 306ms/step 

In [81]:
y_pred_prob = model.predict(X_train, verbose=0)
y_pred = (y_pred_prob > 0.8).astype(int)

metrics_dict = compute_classification_metrics(y_train, y_pred)

results_df = pd.DataFrame([{
    "Accuracy": metrics_dict["Accuracy"],
    "Precision": metrics_dict["Precision"],
    "Recall": metrics_dict["Recall"],
    "F1": metrics_dict["F1-Score"],
    "TP": metrics_dict["TP"],
    "TN": metrics_dict["TN"],
    "FP": metrics_dict["FP"],
    "FN": metrics_dict["FN"],
}])

print("\nTRAINING DATA METRICS")
print("=" * 50)
print(results_df.to_string(index=False))
print("=" * 50)


TRAINING DATA METRICS
 Accuracy  Precision  Recall  F1  TP   TN  FP  FN
  0.88662        0.0     0.0 0.0   0 4168   0 533


In [86]:
y_pred_prob = model.predict(X_test, verbose=0)

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

results_list = []

for t in thresholds:
    y_pred = (y_pred_prob > t).astype(int)
    metrics_dict = compute_classification_metrics(y_test, y_pred)
    
    results_list.append({
        "Threshold": t,
        "Accuracy": metrics_dict["Accuracy"],
        "Precision": metrics_dict["Precision"],
        "Recall": metrics_dict["Recall"],
        "F1": metrics_dict["F1-Score"],
        "TP": metrics_dict["TP"],
        "TN": metrics_dict["TN"],
        "FP": metrics_dict["FP"],
        "FN": metrics_dict["FN"],
    })

results_df = pd.DataFrame(results_list)

print("\nTEST DATA METRICS FOR DIFFERENT THRESHOLDS")
print("=" * 80)
print(results_df.to_string(index=False))
print("=" * 80)


TEST DATA METRICS FOR DIFFERENT THRESHOLDS
 Threshold  Accuracy  Precision   Recall       F1  TP  TN  FP  FN
       0.3  0.419643   0.161151 0.982456 0.276885 112 311 583   2
       0.4  0.531746   0.188153 0.947368 0.313953 108 428 466   6
       0.5  0.729167   0.273504 0.842105 0.412903  96 639 255  18
       0.6  0.859127   0.397059 0.473684 0.432000  54 812  82  60
       0.7  0.890873   0.642857 0.078947 0.140625   9 889   5 105
       0.8  0.886905   0.000000 0.000000 0.000000   0 894   0 114


In [80]:
print(y_train.value_counts())
print(y_test.value_counts())

label
0    4168
1     533
Name: count, dtype: int64
label
0    894
1    114
Name: count, dtype: int64


In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import os

# ==============================
# Directory setup
# ==============================

MODEL_NAME = "BiGRU+CONV"

CSV_PATH = "./csv"
IMAGE_PATH = "./image"

os.makedirs(CSV_PATH, exist_ok=True)
os.makedirs(IMAGE_PATH, exist_ok=True)

METRIC_FILE = os.path.join(CSV_PATH, f"{MODEL_NAME}_metrics.csv")
HISTORY_FILE = os.path.join(CSV_PATH, f"{MODEL_NAME}_training_history.csv")
PLOT_FILE = os.path.join(IMAGE_PATH, f"{MODEL_NAME}_history.png")


# ==============================
# Metrics Class
# ==============================

class ComputeMetrics:
    """Class to compute and manage classification metrics."""

    def __init__(self, y_true, y_pred):
        self.y_true = y_true.flatten() if y_true.ndim > 1 else y_true
        self.y_pred = y_pred.flatten() if y_pred.ndim > 1 else y_pred

    def get_confusion_matrix(self):
        return confusion_matrix(self.y_true, self.y_pred)

    def get_precision(self):
        return precision_score(self.y_true, self.y_pred, zero_division=0)

    def get_recall(self):
        return recall_score(self.y_true, self.y_pred, zero_division=0)

    def get_f1_score(self):
        return f1_score(self.y_true, self.y_pred, zero_division=0)

    def get_accuracy(self):
        return accuracy_score(self.y_true, self.y_pred)

    def get_all_metrics(self):
        cm = self.get_confusion_matrix()
        tn, fp, fn, tp = cm.ravel()

        return {
            "TP": int(tp),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "Precision": float(self.get_precision()),
            "Recall": float(self.get_recall()),
            "F1-Score": float(self.get_f1_score()),
            "Accuracy": float(self.get_accuracy())
        }


# ==============================
# Predict
# ==============================

y_pred = (model.predict(X_test) > 0.3).astype(int)

metrics_calculator = ComputeMetrics(y_test, y_pred)
metrics = metrics_calculator.get_all_metrics()

print("\n" + "="*50)
print("CLASSIFICATION METRICS ON TEST SET")
print("="*50)

for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key:12s}: {value:.4f}")
    else:
        print(f"{key:12s}: {value}")


# ==============================
# Save Metrics
# ==============================

metrics_df = pd.DataFrame([metrics])
metrics_df.insert(0, "Model", MODEL_NAME)
metrics_df.to_csv(METRIC_FILE, index=False)

print(f"\n✓ Metrics saved to: {METRIC_FILE}")


# ==============================
# Plot Training History
# ==============================

print("\n" + "="*50)
print("TRAINING HISTORY VISUALIZATION")
print("="*50)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
axes[0].plot(history.history["accuracy"], label="Train Accuracy", linewidth=2)
axes[0].plot(history.history["val_accuracy"], label="Val Accuracy", linewidth=2)

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Model Accuracy per Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history["loss"], label="Train Loss", linewidth=2)
axes[1].plot(history.history["val_loss"], label="Val Loss", linewidth=2)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Model Loss per Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_FILE, dpi=300, bbox_inches="tight")

print(f"✓ Training history plot saved to: {PLOT_FILE}")

plt.show()


# ==============================
# Save Training History CSV
# ==============================

history_df = pd.DataFrame({
    "epoch": range(1, len(history.history["accuracy"]) + 1),
    "train_accuracy": history.history["accuracy"],
    "val_accuracy": history.history["val_accuracy"],
    "train_loss": history.history["loss"],
    "val_loss": history.history["val_loss"]
})

history_df.to_csv(HISTORY_FILE, index=False)

print(f"✓ Training history saved to: {HISTORY_FILE}")
print("="*50)

In [ ]:
import seaborn as sns
import os
from matplotlib import pyplot as plt

MODEL_NAME = "BiGRU+CONV"

IMAGE_PATH = "./image"
os.makedirs(IMAGE_PATH, exist_ok=True)

CM_FILE = os.path.join(IMAGE_PATH, f"{MODEL_NAME}_confusion_matrix.png")


# ==============================
# Confusion Matrix Visualization
# ==============================

print("\n" + "="*50)
print("CONFUSION MATRIX VISUALIZATION")
print("="*50)

cm = metrics_calculator.get_confusion_matrix()

fig, ax = plt.subplots(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative","Positive"],
    yticklabels=["Negative","Positive"],
    cbar_kws={"label":"Count"},
    annot_kws={"size":14},
    ax=ax
)

ax.set_xlabel("Predicted Label", fontsize=12, fontweight="bold")
ax.set_ylabel("True Label", fontsize=12, fontweight="bold")
ax.set_title(f"Confusion Matrix - {MODEL_NAME}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(CM_FILE, dpi=300, bbox_inches="tight")

print(f"✓ Confusion matrix saved to: {CM_FILE}")

plt.show()


# ==============================
# Metrics Summary Table
# ==============================

print("\n" + "="*50)
print("METRICS SUMMARY TABLE")
print("="*50)

print(metrics_df.to_string(index=False))

print("="*50)